IMPORT DATASET

In [1]:
import pandas as pd

df_first = pd.read_csv("labeling_500.csv")

df_first["sentiment"].value_counts()

sentiment
0    271
2    141
1     88
Name: count, dtype: int64

In [2]:
texts = df_first["processed_text"].tolist()
labels = df_first["sentiment"].tolist()

IMPORT LIBRARY

In [3]:
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from transformers import(
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer

)

SPLIT DATA

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.15,
    random_state=42,
    stratify=labels
)

In [5]:
print(pd.Series(y_train).value_counts())
print(pd.Series(y_test).value_counts())

0    230
2    120
1     75
Name: count, dtype: int64
0    41
2    21
1    13
Name: count, dtype: int64


In [6]:
Model = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(Model)

In [7]:
train_encodings = tokenizer(
    X_train,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    X_test,
    truncation=True,
    padding=True,
    max_length=128
)


In [8]:
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor((self.labels[idx]))

        return item
    
    def __len__(self):
        return len(self.labels)


In [9]:
train_dataset = SentimentDataset(
    train_encodings,
    y_train
)

test_dataset = SentimentDataset(
    test_encodings,
    y_test
)

In [10]:
model = AutoModelForSequenceClassification.from_pretrained(
    Model,
    num_labels=3
)

def compute_metric(eval_pred):
    logits, labels = eval_pred

    prediction = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, prediction)

    return {
        "accuracy":acc
    }

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
training_args = TrainingArguments(
    output_dir="./indobert_sentiment",

    eval_strategy="epoch",

    save_strategy="epoch",

    num_train_epochs=3,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    learning_rate=2e-5,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="accuracy",

    logging_steps=50
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metric
)

In [13]:
trainer.train()

c:\Users\Satri\Downloads\dataset gambar\GITHUB\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.865400,0.876894,0.666667
2,0.462158,1.059727,0.666667
3,0.189710,1.333230,0.640000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Satri\Downloads\dataset gambar\GITHUB\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Satri\Downloads\dataset gambar\GITHUB\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=321, training_loss=0.548719321455911, metrics={'train_runtime': 614.2825, 'train_samples_per_second': 2.076, 'train_steps_per_second': 0.523, 'total_flos': 83867401900800.0, 'train_loss': 0.548719321455911, 'epoch': 3.0})

In [14]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(
    predictions.predictions,
    axis=1
)

c:\Users\Satri\Downloads\dataset gambar\GITHUB\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [15]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "negative",
            "positive",
            "neutral"
        ]
    )
)

              precision    recall  f1-score   support

    negative       0.73      0.93      0.82        41
    positive       1.00      0.15      0.27        13
     neutral       0.48      0.48      0.48        21

    accuracy                           0.67        75
   macro avg       0.74      0.52      0.52        75
weighted avg       0.71      0.67      0.63        75



In [16]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

[[38  0  3]
 [ 3  2  8]
 [11  0 10]]


In [17]:
df_pred = pd.read_csv("2500data_unlabeled.csv")

In [18]:
new_encodings = tokenizer(
    df_pred["processed_text"].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

In [19]:
import torch

class PredictionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

    def __len__(self):
        return len(self.encodings["input_ids"])

In [20]:
pred_dataset = PredictionDataset(new_encodings)

In [21]:
predictions = trainer.predict(pred_dataset)

c:\Users\Satri\Downloads\dataset gambar\GITHUB\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [22]:
import numpy as np

pred_labels = np.argmax(
    predictions.predictions,
    axis=1
)

In [23]:
df_pred["sentiment"] = pred_labels

In [26]:
label_map = {
    0: "negatif",
    1: "positif",
    2: "netral"
}

df_pred["sentiment_text"] = (
    df_pred["sentiment"]
    .map(label_map)
)

In [27]:
df_pred.to_csv(
    "hasil_prediksi_2500_indobert.csv",
    index=False
)

In [28]:
df_pred["sentiment_text"].value_counts()

sentiment_text
negatif    1618
netral      804
positif      78
Name: count, dtype: int64